# Fallbacks for a missing coronal-hole strip (not built)

When the strip (`ch_cm_lead_72h`) is missing, the submitted model uses a submodel without it.
This compares that with filling the strip from older images, from the image one solar rotation
earlier, or from the PFSS ring. Study: `python -m training.ch_fallback`; write-up
`docs/ch_fallback.md`.

In [1]:
import numpy as np
import pandas as pd

from training import ch_fallback as cf
from training import pull_features as pf

d = pd.read_csv(cf.EXPERIMENTS / "ch_fallback.csv")
ref = d[d.variant == "reference"].set_index("fold")
d["vs reference"] = d["all"] / d.fold.map(ref["all"]) - 1
d["affected vs reference"] = d["affected"] / d.fold.map(ref["affected"]) - 1
summary = d.groupby("variant", sort=False).agg(
    vs_reference=("vs reference", lambda s: np.exp(np.log1p(s).mean()) - 1),
    submodel_runs=("submodel runs", "sum"))
summary.round(4)

,vs_reference,submodel_runs
variant,,
reference,0.0000,331
older 48 h,0.0004,313
rotation,0.0004,96
pfss proxy,0.0013,0
older + rotation,0.0026,96
older + rotation + proxy,0.0035,0


## Affected runs (those the reference forecasts with the submodel)

In [2]:
(100 * d.pivot(index="variant", columns="fold", values="affected vs reference")
 .dropna(axis=1)).round(1)

fold,2021,2024
variant,,
older + rotation,-23.0,4.4
older + rotation + proxy,-23.0,5.7
older 48 h,-11.1,-1.3
pfss proxy,-9.5,1.3
reference,0.0,0.0
rotation,-27.1,5.2


## Where the gaps are: usable AIA cut-offs per month (below 90%)

In [3]:
c = pf.load_coronal_holes()
ok = c["ch_cm"].notna()
m = ok.groupby(c.index.tz_convert(None).to_period("M")).mean()
m[m < 0.9].round(2)

cutoff
2019-08    0.80
2022-06    0.86
2024-12    0.00
2025-01    0.78
2025-09    0.35
2025-10    0.00
2025-11    0.00
2025-12    0.49
2026-03    0.62
Freq: M, Name: ch_cm, dtype: float64

The holes are in the SDO browse archive itself (the day directories are empty), so they
can't be refilled from that source; live, the strip was present in every staging cycle since
2026-10-26. None of the stand-ins beats the submodel on the 2024 fold, where 312 of the 331
affected runs are.